# DSC0152 - notebook v63

Target: KS OOF di atas `0.5299`, naik dari rekor sebelumnya `0.528717`.

## Yang berubah dari v62

1. **Feature selection jadi lever utama.** Sweep tiga fold seed (`42, 7, 13`) membandingkan level seleksi dan hyperparameter yang cocok untuknya. Tanpa seleksi KS 0.524794, dengan 200 kolom terpilih KS 0.527077 dengan sebaran antar seed paling kecil dari semua konfigurasi (sd 0.00017).
2. **Hipotesis awal salah, dan itu dicatat.** Dugaan bahwa `feature_fraction` 0.3 terlalu sempit untuk set 200 kolom ternyata tidak berlaku. Menaikkannya ke 0.6 menaikkan KS 0.0003 tetapi menurunkan AUC 0.0004 dan memperbesar sebaran antar seed tujuh kali, jadi 0.3 dipertahankan.
3. **Tangga seleksi sebagai sumbu ekstrapolasi baru.** Dua arah yang terbukti di leaderboard mendorong model berfitur lebih kaya menjauh dari model berfitur lebih sedikit. Ketajaman seleksi adalah ukuran lain dari quantity yang sama, jadi `sel200` didorong menjauh dari `sel400` memakai mekanisme yang persis sama.
4. **Base lintas run.** v62 dan v63 memakai empat fold seed yang sama, jadi prediksinya bisa dirata-ratakan sebelum diekstrapolasi. Karena kedua run menyimpan OOF per fold seed, pemeriksaan ketahanan bisa dilakukan di dalam tiap seed untuk semua kandidat lintas run.

## Hasil

| Kandidat | KS OOF | AUC | Korelasi vs jangkar |
|---|---:|---:|---:|
| `consensus5_best` | **0.529867** | 0.847204 | 0.995281 |
| `best_single` | 0.529767 | 0.847304 | 0.994953 |
| `consensus12` | 0.529620 | 0.847289 | 0.995430 |
| rekor sebelumnya (v62) | 0.528717 | 0.847128 | — |

`consensus5_best` naik `+0.001150` dari rekor sebelumnya. Per fold seed: `[0.52697, 0.52799, 0.52625, 0.52763]`, dan selisih terhadap base tanpa ekstrapolasi positif di 3 dari 4 seed dengan mean `+0.00089`.

## Batas yang jujur

Nilai `0.529867` masih `0.00013` di bawah `0.530000`. Pada 70.000 baris OOF, galat standar KS sekitar 0.002 sampai 0.003, jadi selisih sebesar itu tidak dapat dibedakan dari noise. Angka ini berarti mendekati `0.53` setelah pembulatan, bukan berarti melampaui `0.53`.

## Aturan yang dipatuhi

Python saja, hanya data kompetisi lokal, preprocessing dan feature engineering manual, tanpa AutoML dan AutoFE, tanpa data eksternal. Seleksi fitur dihitung per fold dari probe gain pada baris training fold itu saja. Factor loading dihitung dari label fold training saja. Semua agregat historis dipotong pada `decision_datetime`. Tidak ada push ke GitHub. Semua artefak hanya di `agas/`.

`RUN_TRAINING = False` membuat notebook hanya membaca cache sehingga bisa diverifikasi tanpa menghitung ulang.

In [1]:
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd

RUN_TRAINING = False
FOLDS = 10
FOLD_SEEDS = '42,7,13,21'
LR = 0.01
L2 = 5.0
SEL200_SEEDS = '0,1'

ROOT = Path.cwd()
AGAS = ROOT / 'agas'
VER = AGAS / 'v63'
OUT = VER / 'output'
SHARED = AGAS / 'shared'
CACHE = SHARED / 'cache'
TRAINING = VER / 'training'
V62 = AGAS / 'v62' / 'output'
V61 = AGAS / 'v61' / 'output'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SHARED))
os.chdir(ROOT)
print('numpy', np.__version__, '| pandas', pd.__version__)
print('version dir', VER)

numpy 2.5.1 | pandas 3.0.5
version dir /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v63


## 1. Data lokal

Tujuh tabel kompetisi dipakai. Tidak ada sumber eksternal.

In [2]:
import lightgbm, xgboost, catboost, sklearn

print('lightgbm', lightgbm.__version__)
print('xgboost', xgboost.__version__)
print('catboost', catboost.__version__)
print('sklearn', sklearn.__version__)

apps = pd.read_csv(ROOT / 'data/train_applications.csv')
print('train rows', len(apps), '| base rate', round(apps.target.mean(), 5))

for name in ['train_applications', 'test_applications', 'customer_profiles',
             'previous_applications', 'repayment_history', 'account_behavior',
             'bureau_observations', 'ecommerce_transactions']:
    p = ROOT / 'data' / f'{name}.csv'
    print(f'{name:24s} {p.stat().st_size / 1e6:9.1f} MB')

lightgbm 4.7.0
xgboost 3.3.0
catboost 1.2.10
sklearn 1.9.0
train rows 70000 | base rate 0.13581
train_applications             8.2 MB
test_applications              3.5 MB
customer_profiles              6.4 MB
previous_applications         26.7 MB
repayment_history            159.7 MB
account_behavior             100.0 MB
bureau_observations          114.7 MB
ecommerce_transactions       528.8 MB


## 2. Sweep level seleksi dan hyperparameter

Dua belas konfigurasi, masing-masing rata-rata tiga fold seed `42`, `7`, `13`. Seleksi dilakukan per fold dengan probe gain pada baris training fold tersebut.

In [3]:
sweep = pd.read_csv(OUT / 'sweep_v63.json') if (OUT / 'sweep_v63.json').exists() else None
if sweep is None:
    sweep = json.load(open(SHARED / 'harness_output' / 'sweep_v63.json'))
    sweep = pd.DataFrame(sweep)
sweep_sorted = sweep.sort_values('ks', ascending=False)[['name', 'auc', 'ks', 'ks_sd', 'ks_per_seed']]
sweep_sorted

,name,auc,ks,ks_sd,ks_per_seed
2,t200_ff06,0.845785,0.527346,0.001168,"[0.52718, 0.52885, 0.526]"
12,t200_ff06_l2_2,0.845873,0.527163,0.000609,"[0.5263, 0.52765, 0.52754]"
1,t200_ff03,0.846141,0.527077,0.000172,"[0.52727, 0.52711, 0.52685]"
9,t400_ff04,0.846195,0.525898,0.000372,"[0.52567, 0.52642, 0.5256]"
5,t200_ff06_lv31,0.845619,0.525894,0.000628,"[0.52622, 0.52644, 0.52502]"
8,t300_ff05,0.846150,0.525802,0.000356,"[0.52588, 0.52619, 0.52533]"
6,t200_ff06_lv31_mcs60,0.845622,0.525561,0.000916,"[0.52428, 0.52639, 0.52601]"
13,t200_ff06_lr02,0.845547,0.525390,0.000748,"[0.52575, 0.52607, 0.52435]"
7,t150_ff07,0.845572,0.525297,0.000735,"[0.52481, 0.52475, 0.52634]"
4,t200_ff06_mcs60,0.845739,0.525193,0.000730,"[0.52585, 0.52555, 0.52418]"


In [4]:
if RUN_TRAINING:
    os.system(f'python3 {VER / "sweep_v63.py"}')

print('konfigurasi terbaik menurut KS:')
print(sweep_sorted.head(6).to_string(index=False))

konfigurasi terbaik menurut KS:
          name      auc       ks    ks_sd                 ks_per_seed
     t200_ff06 0.845785 0.527346 0.001168   [0.52718, 0.52885, 0.526]
t200_ff06_l2_2 0.845873 0.527163 0.000609  [0.5263, 0.52765, 0.52754]
     t200_ff03 0.846141 0.527077 0.000172 [0.52727, 0.52711, 0.52685]
     t400_ff04 0.846195 0.525898 0.000372  [0.52567, 0.52642, 0.5256]
t200_ff06_lv31 0.845619 0.525894 0.000628 [0.52622, 0.52644, 0.52502]
     t300_ff05 0.846150 0.525802 0.000356 [0.52588, 0.52619, 0.52533]


## 3. Ensemble

10 fold, empat fold seed, dan kaki-kaki berikut. `selN` berarti N kolom terpilih gain per fold. `feature_fraction` mengikuti hasil sweep: 0.9 untuk 100 kolom, 0.3 untuk 200, 0.5 untuk 300, 0.4 untuk 400.

In [5]:
cmd = (f'python3 {TRAINING / "train_submission_v63.py"} --folds {FOLDS} '
       f'--fold-seeds {FOLD_SEEDS} --lr {LR} --l2 {L2} '
       f'--sel200-seeds {SEL200_SEEDS} --out-prefix v63')
print(cmd)
if RUN_TRAINING:
    os.system(cmd)
else:
    print('dilewati, memakai agas/v63/output/preds_v63.npz yang sudah ada')

python3 /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v63/training/train_submission_v63.py --folds 10 --fold-seeds 42,7,13,21 --lr 0.01 --l2 5.0 --sel200-seeds 0,1 --out-prefix v63
dilewati, memakai agas/v63/output/preds_v63.npz yang sudah ada


In [6]:
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score

z = np.load(OUT / 'preds_v63.npz', allow_pickle=True)
seeds = [int(s) for s in z['seeds']]
train_ids, test_ids = z['train_ids'], z['test_ids']
ymap = apps.set_index('application_id').reindex(pd.Index(train_ids))
y = ymap['target'].values.astype(int)

rows = []
for k in sorted(c for c in z.files if c.startswith('oof_')):
    name = k[4:]
    p = z[k]
    per = [ks_2samp(z[f'oofseed_{name}_{j}'][y == 1], z[f'oofseed_{name}_{j}'][y == 0]).statistic
           for j in range(len(seeds))]
    rows.append({'model': name, 'auc': roc_auc_score(y, p),
                 'ks': ks_2samp(p[y == 1], p[y == 0]).statistic,
                 'ks_per_seed': [round(v, 5) for v in per]})
base = pd.DataFrame(rows).sort_values('ks', ascending=False).reset_index(drop=True)
base

,model,auc,ks,ks_per_seed
0,sel200,0.846981,0.526330,"[0.52548, 0.52613, 0.52452, 0.5232]"
1,sel200b,0.846783,0.525837,"[0.52488, 0.52499, 0.52431, 0.52429]"
2,xgb_full,0.846639,0.524982,"[0.52449, 0.52612, 0.5246, 0.52525]"
3,sel300,0.846793,0.524864,"[0.52515, 0.52498, 0.52286, 0.52361]"
4,cat_full,0.846046,0.524848,"[0.52416, 0.52498, 0.52128, 0.52404]"
5,sel400,0.846730,0.524526,"[0.52423, 0.52556, 0.52338, 0.52385]"
6,lgb_full,0.846502,0.523734,"[0.52278, 0.526, 0.52324, 0.52227]"
7,sel100,0.845208,0.523120,"[0.52259, 0.52234, 0.52219, 0.52315]"
8,lgb_faclean,0.846444,0.523102,"[0.52251, 0.52441, 0.52277, 0.52231]"
9,lgbx_full,0.846305,0.523064,"[0.52284, 0.5243, 0.5225, 0.5212]"


## 4. Pencarian arah dan pemilihan consensus

Grid 1296 baris: empat base, delapan kaki, sembilan rasio untuk sumbu tunggal, dan semua pasangan kaki dengan tiga nilai per sumbu untuk dua sumbu. Arah hanya direkomendasikan bila KS membaik di sedikitnya tiga dari empat fold seed.

In [7]:
g = pd.read_csv(OUT / 'gradient_v63_multirun.csv')
print(f"grid {len(g)} baris | best single {g.ks.max():.6f} | "
      f"robust {int(g.robust.sum())} ({100 * g.robust.mean():.1f}%)")
pd.set_option('display.width', 220)
g.sort_values('ks', ascending=False).head(12).to_string(index=False)

grid 1296 baris | best single 0.529767 | robust 557 (43.0%)


'                                               name          base         leg1            leg2     ratio       ks      auc  ks_base  auc_base  worst_delta  mean_delta  n_pos  robust\n   v63sel_lin0.5|v63:lgb_fulla0.25|v62:lgb_facb0.75 v63sel_lin0.5 v63:lgb_full     v62:lgb_fac 0.25/0.75 0.529767 0.847304 0.526287   0.84745     0.000223    0.001142      4    True\n    v63sel_lin0.5|v63:lgb_fulla0.5|v62:lgb_facb0.75 v63sel_lin0.5 v63:lgb_full     v62:lgb_fac  0.5/0.75 0.529591 0.846934 0.526287   0.84745    -0.000934    0.000422      3    True\n                     v63sel_lin0.5|v62:lgb_fac|r1.0 v63sel_lin0.5  v62:lgb_fac             NaN       1.0 0.529428 0.847337 0.526287   0.84745    -0.001184    0.000651      2   False\n      v63sel_lin0.5|v62:lgb_topa0.5|v62:lgb_facb0.5 v63sel_lin0.5  v62:lgb_top     v62:lgb_fac   0.5/0.5 0.529353 0.846456 0.526287   0.84745    -0.001384   -0.000401      1   False\n    v63sel_lin0.5|v63:lgb_fulla0.25|v62:lgb_facb0.5 v63sel_lin0.5 v63:lgb_full     v

In [8]:
if RUN_TRAINING:
    os.system(f'python3 {TRAINING / "search_multirun_v63.py"}')
    os.system(f'python3 {TRAINING / "final_v63.py"}')

summary = json.load(open(OUT / 'summary_v63_final.json'))
print('ukuran consensus terbaik:', summary['best_consensus_size'])
print('KS per fold seed, consensus:', summary['consensus_per_seed'])
print('KS per fold seed, base    :', [round(v, 5) for v in summary['base_per_seed']])
print('selisih per fold seed     :', [round(v, 5) for v in summary['delta_per_seed']])
pd.DataFrame(summary['size_sweep']).sort_values('ks', ascending=False).to_string(index=False)

ukuran consensus terbaik: 5
KS per fold seed, consensus: [0.52697, 0.52799, 0.52625, 0.52763]
KS per fold seed, base    : [0.52709, 0.52668, 0.52607, 0.52546]
selisih per fold seed     : [-0.00012, 0.00131, 0.00018, 0.00217]


' size       ks      auc                             per_seed\n    5 0.529867 0.847204 [0.52697, 0.52799, 0.52625, 0.52763]\n    6 0.529827 0.847263 [0.52779, 0.52806, 0.52618, 0.52743]\n   12 0.529620 0.847289  [0.52812, 0.52824, 0.5264, 0.52768]\n   10 0.529596 0.847267 [0.52822, 0.52824, 0.52627, 0.52766]\n    4 0.529576 0.847120  [0.52681, 0.5281, 0.52639, 0.52767]\n   11 0.529569 0.847273  [0.52817, 0.52828, 0.52645, 0.5276]\n    7 0.529564 0.847278 [0.52777, 0.52811, 0.52636, 0.52755]\n    8 0.529433 0.847286 [0.52808, 0.52837, 0.52665, 0.52793]\n    3 0.529426 0.847238  [0.5271, 0.52828, 0.52562, 0.52847]\n    2 0.529413 0.847143 [0.52771, 0.52778, 0.52589, 0.52811]\n    9 0.529375 0.847289 [0.52806, 0.52831, 0.52662, 0.52768]'

In [9]:
summary = json.load(open(OUT / 'summary_v63_final.json'))
cands = pd.DataFrame([{k: v for k, v in c.items() if k != 'problems'}
                     for c in summary['candidates']])
cands

,name,ks,auc,spearman_anchor,file,per_seed
0,consensus5_best,0.529867,0.847204,0.995281,submission_v63f_consensus5_best.csv,NaN
1,best_single,0.529767,0.847304,0.994953,submission_v63f_best_single.csv,"[0.52732, 0.52813, 0.52629, 0.52813]"
2,consensus12,0.529620,0.847289,0.995430,submission_v63f_consensus12.csv,NaN
3,consensus7,0.529564,0.847278,0.995514,submission_v63f_consensus7.csv,NaN
4,consensus3,0.529426,0.847238,0.994696,submission_v63f_consensus3.csv,NaN
5,consensus9,0.529375,0.847289,0.995555,submission_v63f_consensus9.csv,NaN
6,anchor35_consensus,NaN,NaN,0.999422,submission_v63f_anchor35_consensus.csv,NaN
7,anchor60_consensus,NaN,NaN,0.998299,submission_v63f_anchor60_consensus.csv,NaN


## 5. Verifikasi tahan banting

`verify_v63_consensus.py` membangun ulang consensus dari kode yang sama lalu membandingkannya dengan file yang sudah ditulis. Kesamaan harus persis, supaya angka di laporan ini benar-benar berasal dari file submission.

In [10]:
os.system(f'python3 {TRAINING / "verify_v63_consensus.py"}')

grid rows 1296 | best KS 0.529767 | robust 557 (43.0%)

consensus members:
  1. v63sel_lin0.5|v63:lgb_fulla0.25|v62:lgb_facb0.75  ks=0.529767
  2. v63sel_lin0.5|v63:lgb_fulla0.5|v62:lgb_facb0.75  ks=0.529591
  3. v63sel_lin0.5|v62:lgb_fac|r1.0  ks=0.529428
  4. v63sel_lin0.5|v62:lgb_topa0.5|v62:lgb_facb0.5  ks=0.529353
  5. v63sel_lin0.5|v63:lgb_fulla0.25|v62:lgb_facb0.5  ks=0.529267

=== per fold seed (seeds [42, 7, 13, 21]) ===
  base v63sel_lin0.5                           ks=0.526287 per_seed=[np.float64(0.52709), np.float64(0.52668), np.float64(0.52607), np.float64(0.52546)]
  v63 consensus                                ks=0.529867 per_seed=[np.float64(0.52697), np.float64(0.52799), np.float64(0.52625), np.float64(0.52763)]

extrapolation delta per seed: [np.float64(-0.00012), np.float64(0.00132), np.float64(0.00018), np.float64(0.00217)]
positive in 3/4 seeds | mean +0.00089 | worst -0.00012

rebuilt vs shipped submission_v63f_consensus5_best.csv spearman: 1.00000000
spearman vs

0

## 6. Validasi file submission

Setiap kandidat diperiksa ulang: 30000 baris, `application_id` unik dan sama dengan `data/test_applications.csv`, probabilitas di rentang [0, 1] tanpa NaN.

In [11]:
ref = set(pd.read_csv(ROOT / 'data/test_applications.csv', usecols=['application_id']).application_id)

rows = []
for path in sorted(OUT.glob('submission_*.csv')):
    d = pd.read_csv(path)
    ok = (len(d) == 30000 and d.application_id.is_unique and set(d.application_id) == ref
          and d.default_probability.between(0, 1).all() and not d.default_probability.isna().any())
    rows.append({'file': path.name, 'rows': len(d), 'ok': ok,
                 'min': round(d.default_probability.min(), 6),
                 'max': round(d.default_probability.max(), 6)})
res = pd.DataFrame(rows)
print('total file:', len(res), '| invalid:', int((~res.ok).sum()))
res

total file: 20 | invalid: 0


,file,rows,ok,min,max
0,submission_v63d_anchor35_consensus.csv,30000,True,0.000033,1.0
1,submission_v63d_anchor60_consensus.csv,30000,True,0.000033,1.0
2,submission_v63d_cand01.csv,30000,True,0.000033,1.0
3,submission_v63d_cand02.csv,30000,True,0.000033,1.0
4,submission_v63d_cand03.csv,30000,True,0.000033,1.0
5,submission_v63d_cand04.csv,30000,True,0.000033,1.0
6,submission_v63d_cand05.csv,30000,True,0.000033,1.0
7,submission_v63d_cand06.csv,30000,True,0.000033,1.0
8,submission_v63d_cand07.csv,30000,True,0.000033,1.0
9,submission_v63d_cand08.csv,30000,True,0.000033,1.0


## 7. Urutan submit

Jangkar tetap `submission_v61_alt_ext0.5_tm30_lean.csv` dengan public `0.56312`.

1. `submission_v63f_consensus5_best.csv`, KS OOF 0.529867, korelasi 0.995281 terhadap jangkar.
2. `submission_v63f_best_single.csv`, KS OOF 0.529767, korelasi 0.994953, dan profil per seed paling seragam.
3. `submission_v63f_consensus12.csv`, KS OOF 0.529620, rata-rata per seed tertinggi sehingga paling stabil.
4. `submission_v63d_cand02.csv` sampai `cand05.csv`, untuk menguji arah yang berbeda.
5. `submission_v63f_anchor35_consensus.csv`, 35 persen consensus baru di atas jangkar, untuk slot yang harus aman.

Korelasi terhadap jangkar sekitar 0.995, cukup jauh dari 0.9996 milik kandidat v62, jadi kandidat ini benar-benar menguji hipotesis baru dan bukan mengulang skor yang sudah diketahui.